In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# consistent plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

In [ ]:
df = pd.read_csv("../data/globalterrorismdb_0718dist.csv", encoding="latin-1", low_memory=False)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

## column Selection

from 135 available columns, we select only those relevant to our fuzzy logic system

**input variables (fuzzy inputs):**
- `nkill` number of fatalities
- `nwound` number of injured
- `attacktype1_txt` type of attack
- `weaptype1_txt` type of weapon used
- `propextent` property damage extent (1=catastrophic, 4=unknown)

**output variable:**
- `success` whether the attack succeeded (0/1) used as ground truth for evaluation

**context columns (for analysis only):**
- `iyear`, `region_txt`, `country_txt`

In [ ]:
cols = [
    "iyear", "region_txt", "country_txt",
    "attacktype1_txt", "weaptype1_txt",
    "nkill", "nwound", "propextent", "success"
]

df = df[cols].copy()
df.head()

## missing value analysis

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

pd.DataFrame({"missing": missing, "percent": missing_pct})

In [ ]:
# nkill and nwound: missing means no reported casualties fill with 0
df["nkill"] = df["nkill"].fillna(0)
df["nwound"] = df["nwound"].fillna(0)

# propextent: 4 = unknown in GTD codebook, fill missing with 4
df["propextent"] = df["propextent"].fillna(4)

# drop rows where attack type is unknown
df = df[df["attacktype1_txt"] != "Unknown"].reset_index(drop=True)

print(f"Rows after cleaning: {len(df):,}")
print(df.isnull().sum())

## distribution of Key Variables

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# cap at 95th percentile for readability
kill_cap = df["nkill"].quantile(0.95)
wound_cap = df["nwound"].quantile(0.95)

df[df["nkill"] <= kill_cap]["nkill"].hist(bins=40, ax=axes[0], color="#e74c3c", edgecolor="white")
axes[0].set_title("Fatalities (nkill) — capped at 95th percentile")
axes[0].set_xlabel("Number of fatalities")

df[df["nwound"] <= wound_cap]["nwound"].hist(bins=40, ax=axes[1], color="#e67e22", edgecolor="white")
axes[1].set_title("Injuries (nwound) — capped at 95th percentile")
axes[1].set_xlabel("Number of injuries")

plt.tight_layout()
plt.show()

print(f"95th percentile nkill: {kill_cap}")
print(f"95th percentile nwound: {wound_cap}")

In [ ]:
atk_counts = df["attacktype1_txt"].value_counts()

plt.figure(figsize=(10, 5))
sns.barplot(x=atk_counts.values, y=atk_counts.index, palette="Reds_r")
plt.title("Attack Type Distribution")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
wpn_counts = df["weaptype1_txt"].value_counts().head(8)

plt.figure(figsize=(10, 5))
sns.barplot(x=wpn_counts.values, y=wpn_counts.index, palette="Oranges_r")
plt.title("Weapon Type Distribution (Top 8)")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
yearly = df.groupby("iyear").size().reset_index(name="count")

plt.figure(figsize=(12, 4))
plt.plot(yearly["iyear"], yearly["count"], color="#c0392b", linewidth=2)
plt.fill_between(yearly["iyear"], yearly["count"], alpha=0.15, color="#c0392b")
plt.title("Number of Terrorist Attacks per Year (1970–2017)")
plt.xlabel("Year")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## EDA Summary

key findings before fuzzy system design:

- dataset contains **around 174,000 valid rows** after removing unknown attack types
- `nkill` and `nwound` are heavily right-skewed most attacks have 0–2 casualties
- the 95th percentile for fatalities is around 10, but max is 1,570 extreme outliers exist
- bombing/explosion dominates at ~50% of all attacks
- explosives are the most common weapon type
- attack frequency peaked around 2014–2015

these distributions directly inform our fuzzy membership function boundaries